# RAG over O2 stores

**Task:** answer questions such as *"What is the phone number of the O2 store in OC Metropole Zličín?"* from the public
[O2 store export](https://www.o2.cz/servis/prodejny-export/export.json).

**Approach:** the smallest pipeline that solves the task – **load → chunk → embed → retrieve → generate** – with every
choice backed by a measurement. All code lives in [`rag.py`](rag.py) (~240 lines incl. the prompt); this notebook walks through it.
Each step ends with a note on what I would change at a larger scale.

**Results** (139 stores, 21 evaluation questions):
* retrieval hit@1: dense 0.94 · plain BM25 0.47 · stemmed BM25 0.94 · **hybrid 1.00**
* answers: **20/21** correct; both assignment questions answered (Zličín: +420 257 210 978; Sušice is not in the data → *"Nevím"*)

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

import time
from datetime import date

import numpy as np
from IPython.display import Markdown, display
from sentence_transformers import SentenceTransformer

from rag import EMBED_MODEL, LLM_MODEL, SYSTEM_PROMPT, Index, ask, build_prompt, load_stores, to_chunk
from evaluate import answer_table, retrieval_table

## 1. Data

In [2]:
stores = load_stores()  # cached in data/export.json; load_stores(refresh=True) downloads the current export
print(f"{len(stores)} stores in {len({s['address']['city'] for s in stores})} cities")
print("missing phone:          ", [s["name"] for s in stores if not s.get("phones")])
print("empty zip / house no.:  ", sum(not s["address"]["zip"] and not s["address"]["houseNumber"] for s in stores))
print("non-generic description:", sum(s["description"] != "Prodejna O2" for s in stores))
print("weekly hour patterns:   ", len({s["openingHours"] for s in stores}))
print("Sušice in the data:     ", any("Sušic" in s["name"] for s in stores))
print("\nraw opening hours:", stores[1]["openingHours"])
print("next days incl. holidays:", [(d["date"], d["from"], d["to"], d["status"]) for d in stores[1]["openingHoursRaw"][:5]])

139 stores in 92 cities
missing phone:           ['OC Novo Plaza, Novodvorská 1800/136, Praha']
empty zip / house no.:   139
non-generic description: 11
weekly hour patterns:    28
Sušice in the data:      False

raw opening hours: Mo 8:30-19:00;Tu 8:30-19:00;We 8:30-19:00;Th 8:30-19:00;Fr 8:30-19:00;Sa 8:30-17:00;Su off
next days incl. holidays: [('2026-11-13', '8:30', '19:00', 'Otevřeno'), ('2026-11-14', '8:30', '17:00', 'Otevřeno'), ('2026-11-15', '', '', 'Zavřeno'), ('2026-11-16', '8:30', '19:00', 'Otevřeno'), ('2026-11-17', '8:30', '17:00', 'Otevřeno')]


**Finding:** the Sušice store from the assignment is **not in the current export** (there is no Tesco in Sušice and no
"Hrádecká 1310"). The correct answer to the second example question is therefore *"I don't know"* – a good test that the
system does not hallucinate.

> **At scale:** ingest with PySpark / Azure Data Factory on a schedule, validate the schema, and re-index only changed stores.

## 2. Chunks – one store = one chunk
A store is a small, self-contained record (~200 characters), so splitting would only hurt. The chunk is readable Czech
text instead of raw JSON, because both the embedding model and the LLM work with natural language. The store name,
city and address come first, because that is what users ask by.

In [3]:
chunks = [to_chunk(s) for s in stores]
by_id = {c.id: c for c in chunks}
for store_id in ["734", "658", "914"]:  # Zličín · Národní třída (holiday override) · Novo Plaza (no phone)
    print(by_id[store_id].text, end="\n\n")

Prodejna O2: OC Metropole Zličín, Řevnická 121/1, Praha
Město: Praha
Adresa: Řevnická 121/1, Praha
Telefon: +420 257 210 978
E-mail: Praha_Zlicin@o2.cz
Otevírací doba: pondělí–neděle 9:00–21:00

Prodejna O2: Národní třída 32/58, Praha
Město: Praha
Adresa: Národní třída 32/58, Praha
Telefon: +420 224 948 467
E-mail: Praha_Narodnitr@o2.cz
Otevírací doba: pondělí–pátek 8:30–19:00, sobota 8:30–17:00, neděle zavřeno
Upravená otevírací doba: úterý 17. 11. 2026 8:30–17:00

Prodejna O2: OC Novo Plaza, Novodvorská 1800/136, Praha
Město: Praha
Adresa: Novodvorská 1800/136, Praha
Telefon: neuveden
E-mail: Praha_Novodvorska@o2.cz
Otevírací doba: pondělí–neděle 9:00–21:00



## 3. Index – brute force over a NumPy matrix
Each chunk becomes a normalized 768-d vector from the local model `multilingual-e5-base` (with the `passage:` / `query:`
prefixes the model was trained with). Search is a single matrix–vector product: exact, no infrastructure.

In [4]:
index = Index(chunks, SentenceTransformer(EMBED_MODEL))  # vectors cached in data/vectors_<hash>.npy
query = index.model.encode("query: telefon Zličín", normalize_embeddings=True)
start = time.perf_counter()
for _ in range(1000):
    np.argsort(-(index.vectors @ query))[:3]
print(f"matrix {index.vectors.shape}, one brute-force search: {(time.perf_counter() - start):.3f} ms")

matrix (139, 768), one brute-force search: 0.016 ms


> **At scale:** brute force grows linearly – measured on this laptop ≈ 8 ms per 100k vectors, ≈ 80 ms and 3 GB per 1M.
> From roughly hundreds of thousands of chunks I would switch to an ANN index (**HNSW**, ~log N comparisons), e.g. in
> **Azure AI Search**, which also gives persistence, metadata filters and incremental updates. For 139 stores it would add
> complexity without any measurable gain.

## 4. Retrieval – dense vs. BM25 vs. hybrid
* **Dense** (cosine similarity of E5 vectors) understands paraphrases and English questions, but is weak on inflected proper names.
* **BM25** matches exact tokens (names, street numbers). Plain BM25 fails on Czech inflection (*Zličín* vs. *na Zličíně*),
  so tokens are lower-cased, stripped of diacritics, stopwords removed and truncated to 5 characters – a cheap Czech stemmer.
* **Hybrid** fuses both rankings with **Reciprocal Rank Fusion**: `score = Σ 1 / (60 + rank)` – no score calibration needed.

In [5]:
questions = [
    "Jaké je telefonní číslo O2 prodejny v OC Metropole Zličín?",
    "Jaká je otevírací doba O2 prodejny v OC Tesco, Hrádecká 1310, Sušice?",
    "Jaký je e-mail na prodejnu O2 na Zličíně?",
]
for q in questions:
    print(q)
    for method in ["dense", "bm25_plain", "bm25", "hybrid"]:
        print(f"  {method:10s}", " | ".join(f"{c.name[:34]} ({s:.3f})" for c, s in index.search(q, 3, method)))
    print()

Jaké je telefonní číslo O2 prodejny v OC Metropole Zličín?
  dense      OC Metropole Zličín, Řevnická 121/ (0.916) | OC Kaufland, Leštinská 2336/2, Záb (0.884) | OC Kaufland, Katovická 1404, Strak (0.880)
  bm25_plain OC Metropole Zličín, Řevnická 121/ (12.070) | OC AUPARK, Gočárova třída 1754/48a (6.657) | OC Tesco, Štemberkova 1698, Rychno (6.469)
  bm25       OC Metropole Zličín, Řevnická 121/ (14.126) | OC AUPARK, Gočárova třída 1754/48a (4.992) | OC Olympia Teplice, Srbická 464, T (4.982)
  hybrid     OC Metropole Zličín, Řevnická 121/ (0.033) | OC Kaufland, Katovická 1404, Strak (0.031) | OC Olympia, U Dálnice 744, Brno (0.029)

Jaká je otevírací doba O2 prodejny v OC Tesco, Hrádecká 1310, Sušice?
  dense      OC Tesco, Lidická 3341/137, Břecla (0.890) | OC Galerie Tesco, Sjízdná 5554/2,  (0.889) | OC Tesco, Štemberkova 1698, Rychno (0.878)
  bm25_plain OC Tesco, Štemberkova 1698, Rychno (12.870) | OC Globus, Hradecká 408/40, Brno (11.577) | Jungmannova 1310/10, Kyjov (9.978)
  b

For Sušice every method returns *something* (other Tesco malls, "Hradecká" street in Brno, house number 1310 in Kyjov) –
retrieval always returns its top-k. Rejecting a near-miss is the job of the prompt.

> **At scale:** Azure AI Search runs BM25 + vector search + RRF natively (hybrid query), optionally with a semantic reranker.

## 5. Generation
The prompt gives the model a procedure (identify the store → check it really matches → take the field → answer) and rules
for edge cases: exact *"Nevím…"* when the store is missing, all stores when several match, a source line only when the answer
is known. **Today's and tomorrow's dates are computed in code**, because a 4B model cannot do calendar arithmetic reliably.
The LLM is any OpenAI-compatible endpoint – locally `qwen3:4b-instruct` via Ollama.

In [6]:
print(build_prompt(questions[0], [c for c, _ in index.search(questions[0], 2)], date.today()))

Dnes je neděle 15. 11. 2026, zítra je pondělí 16. 11. 2026.

KONTEXT:
[1]
Prodejna O2: OC Metropole Zličín, Řevnická 121/1, Praha
Město: Praha
Adresa: Řevnická 121/1, Praha
Telefon: +420 257 210 978
E-mail: Praha_Zlicin@o2.cz
Otevírací doba: pondělí–neděle 9:00–21:00

[2]
Prodejna O2: OC Kaufland, Katovická 1404, Strakonice I.
Město: Strakonice I.
Adresa: Katovická 1404, Strakonice I.
Telefon: +420 383 321 963
E-mail: Strakonice@o2.cz
Otevírací doba: pondělí–neděle 8:00–20:00

DOTAZ: Jaké je telefonní číslo O2 prodejny v OC Metropole Zličín?


In [7]:
print(f"LLM: {LLM_MODEL}\n")
for q in questions[:2] + ["Jaká je otevírací doba O2 prodejny v OC Metropole Zličín zítra?"]:
    answer, hits = ask(q, index)
    print("Q:", q, "\nA:", answer, "\n")

LLM: qwen3:4b-instruct

Q: Jaké je telefonní číslo O2 prodejny v OC Metropole Zličín? 
A: Telefonní číslo O2 prodejny v OC Metropole Zličín je +420 257 210 978.  
Zdroj: OC Metropole Zličín, Řevnická 121/1, Praha 

Q: Jaká je otevírací doba O2 prodejny v OC Tesco, Hrádecká 1310, Sušice? 
A: Nevím, tuto informaci v dostupných datech nemám. 

Q: Jaká je otevírací doba O2 prodejny v OC Metropole Zličín zítra? 
A: Zítra (pondělí 16. 11.) má prodejna O2 v OC Metropole Zličín otevírací dobu 9:00–21:00.  
Zdroj: OC Metropole Zličín, Řevnická 121/1, Praha 



> **At scale:** Azure OpenAI (set `LLM_BASE_URL`, `LLM_MODEL`, `LLM_API_KEY`) for better Czech and lower latency than a 4B
> model on CPU. Questions that aggregate over many stores (*"which stores in Brno are open on Sunday?"*) do not fit top-k
> retrieval – they need structured filtering or **function calling** over the store table.

## 6. Evaluation
21 hand-written questions (`eval/questions.json`): both assignment questions, phone / e-mail / hours, inflection, no diacritics,
English, a store without a phone, a holiday, "tomorrow", and 4 questions outside the data.

In [8]:
display(Markdown(retrieval_table(index)))

| method | hit@1 | hit@3 | not at rank 1 |
|---|---|---|---|
| dense | 0.94 | 1.00 | email_locative |
| bm25_plain | 0.47 | 0.59 | email_locative, hours_street, hours_sunday, hours_locative, phone_city_two_stores, hours_holiday, hours_weekend, email_colloquial, email_locative_mall |
| bm25 | 0.94 | 1.00 | hours_street |
| hybrid | 1.00 | 1.00 | – |

17 questions with a gold store; out-of-data questions excluded.

In [9]:
display(Markdown(answer_table(index)))  # full pipeline, ~5 min on CPU

Answer accuracy: **20/21** (keyword check, review answers manually)

| question | ok | answer |
|---|---|---|
| task_phone_cz | ✅ | Telefonní číslo O2 prodejny v OC Metropole Zličín je +420 257 210 978.   Zdroj: OC Metropole Zličín, Řevnická 121/1, Praha |
| task_phone_en | ✅ | Telefonní číslo prodejny O2 v OC Metropole Zličín je +420 257 210 978.   Zdroj: OC Metropole Zličín, Řevnická 121/1, Praha |
| task_hours_cz | ✅ | Nevím, tuto informaci v dostupných datech nemám. |
| task_hours_en | ✅ | Nevím, tuto informaci v dostupných datech nemám. |
| email_locative | ✅ | E-mail na prodejnu O2 na Zličíně je Praha_Zlicin@o2.cz.   Zdroj: OC Metropole Zličín, Řevnická 121/1, Praha |
| hours_street | ❌ | Dnes je neděle 15. 11. 2026, protože je neděle, má prodejna O2 na Národní třídě v Praze zavřeno.   Zdroj: Národní třída 32/58, Praha |
| phone_colloquial | ✅ | Do O2 prodejny v Rychnově nad Kněžnou (OC Tesco) můžete zavolat na číslo +420 494 661 122.   Zdroj: OC Tesco, Štemberkova 1698, Rychnov nad Kněžnou |
| phone_no_diacritics | ✅ | Telefon na O2 v NC Westfield Chodov je +420 220 921 743.   Zdroj: NC Westfield Chodov, Roztylská 2321/19, Praha |
| hours_sunday | ✅ | Prodejna O2 v Šumperku v neděli zavřeno má.   Zdroj: Langrova 2813/4, Šumperk |
| email_en | ✅ | E-mail adresa prodejny O2 v OC Olympia, Plžen je Plzen_Olympia@o2.cz.   Zdroj: OC Olympia, Písecká 972/1, Plzeň |
| note_landmark | ✅ | Telefon prodejny O2 u zastávky šaliny Zelný trh má číslo +420 542 211 120.   Zdroj: Masarykova 12, Brno |
| hours_locative | ✅ | Otevírací doba O2 v Kolíně v Kauflandu (Benešova 1000) je pondělí–neděle 8:00–20:00.   Zdroj: OC Kaufland, Benešova 1000, Kolín |
| phone_city_two_stores | ✅ | Telefon na O2 prodejnu v Kladně v OC Central je +420 312 243 292.   Zdroj: OC Central, Petra Bezruče 3388, Kladno |
| phone_missing | ✅ | Telefonní číslo prodejny O2 v OC Novo Plaza je neuveden.   Zdroj: OC Novo Plaza, Novodvorská 1800/136, Praha |
| hours_holiday | ✅ | 17. listopadu má prodejna O2 v Prostějově zavřeno.   Zdroj: náměstí TGM 27, Prostějov |
| hours_weekend | ✅ | O2 na Václavském náměstí (Václavské náměstí 813/57, Praha) má otevřeno v víkendu (sobota a neděle) 9:00–17:00.   Zdroj: Václavské náměstí 813/57, Praha |
| hours_tomorrow | ✅ | Zítra (pondělí 16. 11.) má prodejna O2 v OC Metropole Zličín otevírací dobu 9:00–21:00.   Zdroj: OC Metropole Zličín, Řevnická 121/1, Praha |
| email_colloquial | ✅ | E-mail na O2 v Ostravě v Nové Karolíně je Ostrava_Karolina@o2.cz.   Zdroj: OC Forum Nová Karolina, Jantarová 3344/4, Ostrava |
| email_locative_mall | ✅ | E-mail na O2 v Galerii Teplice je Teplice_Galerie@o2.cz.   Zdroj: OC Galerie Teplice, náměstí Svobody 3316, Teplice |
| out_of_data_city | ✅ | Nevím, tuto informaci v dostupných datech nemám. |
| out_of_domain | ✅ | Nevím, tuto informaci v dostupných datech nemám. |

The answer check is a keyword match, so I also read every answer. Caveats: the set is small and I tuned the stemmer and
the prompt on the same questions, so the numbers are optimistic.

> **At scale:** a larger labelled set with a held-out split, LLM-as-judge for answer correctness, and tracking of retrieval
> and answer metrics on every data or prompt change.

## Summary – what I built and what I would build next

| Step | This prototype | When it stops being enough | Next step |
|---|---|---|---|
| Ingest | JSON download + local cache | many sources, frequent changes | PySpark / ADF pipeline, scheduled incremental re-index |
| Chunks | 1 store = 1 readable chunk | long documents | structure-aware splitting with overlap |
| Index | NumPy matrix, brute force | ~10⁵–10⁶ vectors, filters, concurrent users | HNSW in Azure AI Search |
| Retrieval | dense + stemmed BM25, RRF | ambiguous queries | semantic reranker (cross-encoder) |
| Generation | local 4B model via Ollama | quality, latency, many users | Azure OpenAI; function calling for aggregations |
| Evaluation | 21 questions, hit@k, keyword check | regressions over time | held-out set, LLM-as-judge, CI |